# Fermi-Hubbard in the Jordan-Wigner spin picture

A compact, end-to-end demonstration of Pepsy's Jordan-Wigner (bosonic) workflow
for the spinful Fermi-Hubbard model, using **one** Jordan-Wigner conversion for
both the Hamiltonian MPO and the time-evolution gates:

1. build the `U1xU1` Fermi-Hubbard Hamiltonian on a chain;
2. ground state via `SymHamiltonian.to_mpo` + `SymDMRG2`;
3. `jw_bond_layout` to see which bonds are two-site-gate-able;
4. imaginary-time evolution with `jw_trotter_gates`, read out with `jw_energy`
   (it converges to the DMRG ground energy);
5. a real-time quench measuring the average double occupancy.


In [1]:
import numpy as np
import pepsy
from pepsy.tensors import SymMPS, SymHamiltonian, site_charge_from_occupations

# Spinful Fermi-Hubbard on an L-site chain, U1xU1 (per-spin particle number).
L, t, U = 4, 1.0, 8.0
edges = [(i, i + 1) for i in range(L - 1)]
occ = [(1, 0), (0, 1)] * (L // 2)          # half filling, Sz = 0
sc = site_charge_from_occupations(occ)

ham = SymHamiltonian.from_edges("fermi_hubbard_u1u1", "U1U1", edges, t=t, U=U)

# One Jordan-Wigner conversion feeds BOTH the MPO and the gates.
mpo = ham.to_mpo(L=L, compress=False)

# Every chain bond is nearest-neighbour, so each is a valid two-site JW gate.
layout = ham.jw_bond_layout()
print("adjacent bonds  :", layout["adjacent"])
print("long-range bonds:", layout["long_range"])

# Ground state via symmetric DMRG (Jordan-Wigner bosonic representation).
gs = SymMPS.for_model("fermi_hubbard_u1u1", L, bond_dim=1, site_charge=sc,
                      seed=1, dtype="complex128")
opt = pepsy.SymDMRG2(mpo, gs, bond_dims=[1, 2, 4, 8, 16], cutoffs=[1e-10],
                     mixer="density_matrix", compute_initial_energy=False)
opt.solve(max_sweeps=30, sweep_sequence="RL", tol=1e-11)
E_gs = float(opt.energy)
print(f"SymDMRG2 ground energy: {E_gs:.8f}")


adjacent bonds  : [(0, 1), (1, 2), (2, 3)]
long-range bonds: []


SymDMRG2 ground energy: -1.11717241


## Imaginary-time evolution converges to the ground state

`ham.jw_trotter_gates(dt, imaginary=True)` builds the bosonic Jordan-Wigner
Trotter step from the *same* conversion as the MPO, and `ham.jw_energy(state)`
reads the energy back by summing local term expectations.


In [2]:
# Imaginary-time evolution in the Jordan-Wigner (bosonic spin) picture.
psi = SymMPS.for_model("fermi_hubbard_u1u1", L, bond_dim=1, site_charge=sc,
                       seed=2, dtype="complex128", fermionic=False)
step = ham.jw_trotter_gates(0.05, imaginary=True, order=2)

for k in range(200):
    psi.apply_gates(step, method="direct", max_bond=16, cutoff=1e-10, normalize=True)
    if (k + 1) % 50 == 0:
        print(f"step {k + 1:3d}  E = {ham.jw_energy(psi):.8f}")

print(f"\nJW imaginary-time energy: {ham.jw_energy(psi):.8f}")
print(f"SymDMRG2 ground energy  : {E_gs:.8f}")
print(f"difference              : {abs(ham.jw_energy(psi) - E_gs):.2e}")


step  50  E = -1.04246841


step 100  E = -1.09909621


step 150  E = -1.11315117


step 200  E = -1.11623032

JW imaginary-time energy: -1.11623032
SymDMRG2 ground energy  : -1.11717241
difference              : 9.42e-04


## Real-time quench: average double occupancy

Starting from the Neel product state (no double occupancy), real-time hopping
builds up double occupancy. Local observables use the state's symmetry-aware
`measure`.


In [3]:
from pepsy.tensors import symm_operator_from_dense, default_physical_sectors

# Double-occupancy projector |up,down><up,down| (basis index 3).
double_op = symm_operator_from_dense(
    np.diag([0.0, 0.0, 0.0, 1.0]).astype("complex128"),
    default_physical_sectors(model="fermi_hubbard_u1u1"),
    symmetry="U1U1", charge=(0, 0), fermionic=False, sites=1,
)

def mean_double_occ(state):
    return float(np.mean([complex(state.measure(double_op, s)).real for s in range(L)]))

psi_t = SymMPS.for_model("fermi_hubbard_u1u1", L, bond_dim=1, site_charge=sc,
                         seed=5, dtype="complex128", fermionic=False)
rt = ham.jw_trotter_gates(0.05, order=2)   # real-time step

print(" t     <double occ>")
print(f"{0.0:.2f}   {mean_double_occ(psi_t):.5f}")
for k in range(30):
    psi_t.apply_gates(rt, method="direct", max_bond=16, cutoff=1e-10)
    if (k + 1) % 10 == 0:
        print(f"{(k + 1) * 0.05:.2f}   {mean_double_occ(psi_t):.5f}")


 t     <double occ>
0.00   0.00000
0.50   0.04336


1.00   0.04244


1.50   0.04314
